In [3]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

In [2]:
# Load existing chunks and embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)

# pull out the raw chunks + their embeddings for clustering
raw = vectorstore.get(include=["documents", "embeddings", "metadatas"])
leaf_texts = raw["documents"]
leaf_embeddings = raw["embeddings"]

print(f"{len(leaf_texts)} leaf chunks loaded")

#llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

979 leaf chunks loaded


In [3]:
# Clustering (Gaussian Mixture Models, the original RAPTOR approach)
%pip install -q scikit-learn umap-learn

import numpy as np
import umap
from sklearn.mixture import GaussianMixture


def reduce_dimensions(embeddings: np.ndarray, n_components: int = 10) -> np.ndarray:
    """UMAP reduction before clustering — GMM struggles in raw high-dim space."""
    reducer = umap.UMAP(n_components=n_components, metric="cosine")
    return reducer.fit_transform(embeddings)


def get_optimal_clusters(embeddings: np.ndarray, max_clusters: int = 20) -> int:
    """BIC-based selection of cluster count, capped by available points."""
    max_clusters = min(max_clusters, len(embeddings) - 1)
    bics = []
    n_range = range(1, max_clusters + 1)
    for n in n_range:
        gm = GaussianMixture(n_components=n, random_state=42)
        gm.fit(embeddings)
        bics.append(gm.bic(embeddings))
    return n_range[np.argmin(bics)]


def cluster_embeddings(embeddings: np.ndarray, threshold: float = 0.1) -> list[list[int]]:
    """Soft-clusters embeddings; a point can belong to multiple clusters
    if its probability for more than one exceeds `threshold`."""
    reduced = reduce_dimensions(embeddings)
    n_clusters = get_optimal_clusters(reduced)
    gm = GaussianMixture(n_components=n_clusters, random_state=42)
    gm.fit(reduced)
    probs = gm.predict_proba(reduced)

    clusters = [[] for _ in range(n_clusters)]
    for point_idx, point_probs in enumerate(probs):
        for cluster_idx, p in enumerate(point_probs):
            if p > threshold:
                clusters[cluster_idx].append(point_idx)
    return [c for c in clusters if c]  # drop empty clusters

def build_summary_input(cluster_texts: list[str], max_chars: int = 6000) -> str:
    joined_parts = []
    total_len = 0
    dropped = 0
    for text in cluster_texts:
        if total_len + len(text) > max_chars:
            dropped += 1
            continue
        joined_parts.append(text)
        total_len += len(text)
    if dropped:
        print(f"  (dropped {dropped}/{len(cluster_texts)} chunks from this cluster's summary input — over char budget)")
    return "\n\n---\n\n".join(joined_parts)

def get_optimal_clusters(embeddings: np.ndarray, max_clusters: int = 20) -> int:
    max_clusters = min(max_clusters, len(embeddings) - 1, 20)
    max_clusters = max(max_clusters, 1)  # never go below 1
    if max_clusters == 1:
        return 1

    bics = []
    n_range = range(1, max_clusters + 1)
    for n in n_range:
        try:
            gm = GaussianMixture(n_components=n, random_state=42, reg_covar=1e-4)
            gm.fit(embeddings.astype(np.float64))
            bics.append(gm.bic(embeddings.astype(np.float64)))
        except ValueError:
            bics.append(np.inf)  # penalize configurations that fail to fit
    return n_range[np.argmin(bics)]


#def cluster_embeddings(embeddings: np.ndarray, threshold: float = 0.1) -> list[list[int]]:
#    reduced = reduce_dimensions(embeddings)
#    n_clusters = get_optimal_clusters(reduced)

#    gm = GaussianMixture(n_components=n_clusters, random_state=42, reg_covar=1e-4)
#    gm.fit(reduced.astype(np.float64))
#    probs = gm.predict_proba(reduced.astype(np.float64))

#    clusters = [[] for _ in range(n_clusters)]
#    for point_idx, point_probs in enumerate(probs):
#        for cluster_idx, p in enumerate(point_probs):
#            if p > threshold:
#                clusters[cluster_idx].append(point_idx)
#    return [c for c in clusters if c]

Note: you may need to restart the kernel to use updated packages.


/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Summarization prompt for a cluster

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    "You are summarizing a cluster of related passages from technical "
    "papers on physics-informed machine learning. Write a concise "
    "summary that captures the shared themes and key information "
    "across all the passages below, in the same technical register "
    "as the source material.\n\n"
    "Passages:\n{passages}\n\n"
    "Summary:"
)

summary_chain = SUMMARY_PROMPT | llm | StrOutputParser()

In [6]:
# One recursive level of the tree
from rag_lab.utils import call_with_backoff, normalize_text

def build_tree_level(texts, text_embeddings, level, checkpoint_path=None):
    clusters = cluster_embeddings(text_embeddings)
    print(f"Level {level}: {len(texts)} nodes -> {len(clusters)} clusters")

    summary_texts, metadata_records = [], []
    start_idx = 0
    if checkpoint_path and checkpoint_path.exists():
        with open(checkpoint_path) as f:
            saved = json.load(f)
        summary_texts = saved["summary_texts"]
        metadata_records = saved["metadata_records"]
        start_idx = len(summary_texts)
        print(f"Resuming from checkpoint: {start_idx}/{len(clusters)} clusters already done")

    for cluster_idx in range(start_idx, len(clusters)):
        member_indices = clusters[cluster_idx]
        cluster_texts = [texts[i] for i in member_indices]
        #joined = "\n\n---\n\n".join(cluster_texts)
        joined = build_summary_input(cluster_texts, max_chars=6000)
        
        summary = normalize_text(
            call_with_backoff(lambda: summary_chain.invoke({"passages": joined}))
        )        
        
        summary_texts.append(summary)
        metadata_records.append({
            "level": level, "cluster_id": cluster_idx,
            "num_source_nodes": len(member_indices),
        })

        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump({"summary_texts": summary_texts, "metadata_records": metadata_records}, f)

    # the embedding call at the end of each level also needs protection
    summary_embeddings = call_with_backoff(
        lambda: np.array(embeddings.embed_documents(summary_texts))
    )
    return summary_texts, summary_embeddings, metadata_records

In [36]:
# Building the full tree, persist every level

import pickle
import time

MAX_LEVELS = 3
MIN_NODES_TO_CONTINUE = 4  # stop recursing once a level is this small

all_texts = list(leaf_texts)
all_embeddings = list(leaf_embeddings)
all_metadata = [{"level": 0, "cluster_id": None, "num_source_nodes": 1} for _ in leaf_texts]

current_texts = leaf_texts
current_embeddings = np.array(leaf_embeddings)

for level in range(1, MAX_LEVELS + 1):
    if len(current_texts) <= MIN_NODES_TO_CONTINUE:
        print(f"Stopping at level {level - 1}: only {len(current_texts)} nodes remain")
        break

    summary_texts, summary_embeddings, metadata = build_tree_level(
        current_texts, current_embeddings, level
    )

    all_texts.extend(summary_texts)
    all_embeddings.extend(summary_embeddings)
    all_metadata.extend(metadata)

    current_texts = summary_texts
    current_embeddings = summary_embeddings

# persist the full multi-level tree
raptor_data = {
    "texts": all_texts,
    "embeddings": all_embeddings,
    "metadata": all_metadata,
}

with open(project_root / "data" / "raptor_tree.pkl", "wb") as f:
    pickle.dump(raptor_data, f)

print(f"\nTotal nodes across all levels: {len(all_texts)}")

Level 1: 979 nodes -> 20 clusters
  (dropped 50/56 chunks from this cluster's summary input — over char budget)
  (dropped 31/36 chunks from this cluster's summary input — over char budget)
  (dropped 47/53 chunks from this cluster's summary input — over char budget)
  (dropped 41/48 chunks from this cluster's summary input — over char budget)
  (dropped 180/187 chunks from this cluster's summary input — over char budget)
  (dropped 32/39 chunks from this cluster's summary input — over char budget)
  (dropped 96/101 chunks from this cluster's summary input — over char budget)
  (dropped 36/40 chunks from this cluster's summary input — over char budget)
  (dropped 41/44 chunks from this cluster's summary input — over char budget)
  (dropped 23/28 chunks from this cluster's summary input — over char budget)
  (dropped 16/20 chunks from this cluster's summary input — over char budget)
  (dropped 125/132 chunks from this cluster's summary input — over char budget)
  (dropped 23/29 chunks f

In [37]:
# Building a separate Chroma collection for the RAPTOR tree
from langchain_core.documents import Document

raptor_documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_texts, all_metadata)
]

raptor_vectorstore = Chroma.from_documents(
    documents=raptor_documents,
    embedding=embeddings,
    persist_directory=str(project_root / "data" / "chroma_raptor")
)
print("RAPTOR tree persisted:", raptor_vectorstore._collection.count(), "nodes")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [ ]:
# Testing

from rag_lab.strategies.raptor import RaptorStrategy

strategy = RaptorStrategy(raptor_vectorstore, llm=llm)

specific_q = "What k-factor did Decke et al. use for ChebConv layers?"
conceptual_q = "Why is combining PINNs and GNNs considered a promising research direction?"

for q in [specific_q, conceptual_q]:
    docs = strategy.retrieve(q)
    levels = [d.metadata.get("level") for d in docs]
    print(f"Query: {q}\nRetrieved levels: {levels}\n")